# TALENTIQ — WHAT THE TRAINING PIPELINE ACTUALLY DID


### this notebook does not train anything by itself
the actual training already happened inside src/train.py when python main.py --stage train was run. here we are just loading what it produced and walking through every step, so anyone reading this can understand the full pipeline without going line by line through the source code.


## STEP 1 — WHY WE NEED A PIPELINE AT ALL


### -- the problem with doing things manually
if you encode and scale the data first, then split into train and test, the test set has already influenced the encoding. the scaler saw the test data when it computed the mean and standard deviation, so the test set is no longer truly unseen. this is called data leakage, and it makes your evaluation numbers look better than they would be in the real world.

the correct order is always: split first, then fit the encoder and scaler only on the training data, then apply the same fitted objects to the test data. a ColumnTransformer makes this easy because you fit it once on train and call transform on test, and the same fitted state is used both times.


## STEP 2 — WHAT THE COLUMNTRANSFORMER DOES


### -- three different transformations, applied to three different groups of columns

OrdinalEncoder on education_level and university_tier — these two columns have a clear order to them. a PhD is more than a Masters, Tier 1 is better than Tier 3. OrdinalEncoder maps each value to a number that preserves this order, for example High School=1, Bachelor=2, Master=3, PhD=4. the exact mapping comes from features.yaml so it is consistent everywhere.

OneHotEncoder on company_type — this column has no natural order. a startup is not more or less than an MNC in any ranked sense. OneHotEncoder gives each category its own 0 or 1 column so the model never assumes one category is mathematically bigger than another.

StandardScaler on all numerical columns — columns like cgpa go up to 10 while experience_years can go up to 30. if we leave them on different scales, the model might think experience_years matters more just because its numbers are bigger. StandardScaler brings every numerical column to mean=0 and standard deviation=1 so they are all on the same playing field.

the engineered features (EmployabilityScore, PortfolioStrength and the rest) pass through remainder=passthrough without any extra transformation, since they were already built from the scaled raw values.


In [ ]:
import sys
sys.path.append('..')
import joblib
import pandas as pd
import numpy as np

preprocessor = joblib.load("../artifacts/preprocessor.pkl")

print("ColumnTransformer — what each transformer handles:")
print()
for name, transformer, columns in preprocessor.transformers:
    print(f"  {name}")
    print(f"    transformer : {transformer.__class__.__name__}")
    print(f"    columns     : {columns}")
    print()


## STEP 3 — WHY SMOTE AND WHY AFTER THE SPLIT


### -- the class imbalance problem
our dataset has about 2.4x more hired candidates than not-hired. a model could score 70% accuracy by just predicting hired for every single row, without learning anything real. this is the imbalance problem.

### -- what SMOTE does
SMOTE stands for Synthetic Minority Oversampling Technique. instead of just copying the minority class rows, it creates brand new synthetic rows that sit between existing minority rows in feature space. the result is a balanced training set where the model sees roughly equal numbers of hired and not-hired examples.

### -- why SMOTE must come after the split, not before
if we applied SMOTE before splitting, some of the synthetic rows generated from a training candidate might end up in the test set. the test set would then contain data that was derived from training data, which is another form of data leakage. SMOTE must only ever touch the training data.

### -- why SMOTE also comes after the ColumnTransformer
SMOTE works by computing distances between rows and interpolating between them. it needs all columns to be numbers. if we ran SMOTE on raw data with text columns like education_level, it would crash. so the order is: encode and scale first, then SMOTE.


In [ ]:
y_train = pd.read_csv("../data/splits/y_train.csv")

print("Class distribution in raw training set (before SMOTE):")
print(y_train.value_counts())
print()
print("After SMOTE, both classes were balanced to 112,970 rows each (225,940 total).")
print("The test set was never touched by SMOTE — it stays as the original 40,000 rows.")


## STEP 4 — HYPERPARAMETER TUNING


### -- what hyperparameters are
every model has settings that you choose before training, not settings the model learns from data. for Logistic Regression, one of these is C, which controls how strictly the model fits the training data. for Random Forest, it is things like how many trees to use and how deep each tree can go. these are called hyperparameters.

### -- why we need to tune them
the default values that come with sklearn are reasonable starting points but they are not the best values for every dataset. tuning means trying a range of values and picking whichever combination gives the best score on the validation set.

### -- GridSearchCV vs RandomizedSearchCV
GridSearchCV tries every single combination in the search space. for Logistic Regression this is manageable because the search space is small.
RandomizedSearchCV randomly samples a fixed number of combinations from the search space. for Random Forest and XGBoost the search space is much larger, so trying every combination would take too long. sampling 10 combinations with cv=4 gives a good enough estimate of which region of the space works best.

### -- what cv=4 means
cv=4 means 4-fold cross validation. for each combination of hyperparameters, the training data is split into 4 parts. the model trains on 3 parts and validates on the remaining 1, repeated 4 times so every part gets a turn as the validation set. the 4 scores are averaged to give a reliable estimate of how well that combination generalises.

### -- what scoring=f1_macro means
we are using F1-macro as the scoring metric instead of accuracy. accuracy is misleading on imbalanced datasets because a model that always predicts the majority class looks accurate. F1-macro computes the F1 score for each class separately and averages them, so it equally penalises the model for getting either class wrong.


In [ ]:
from src.config_loader import load_hyperparameters

hp = load_hyperparameters()

for model_name, config in hp.items():
    search_type = config["search"]
    cv = config["cv"]
    scoring = config["scoring"]
    
    if search_type == "grid":
        param_key = "param_grid"
        n_combinations = 1
        for values in config[param_key].values():
            n_combinations *= len(values)
        total_fits = n_combinations * cv
    else:
        param_key = "param_distributions"
        n_combinations = config["n_iter"]
        total_fits = n_combinations * cv
    
    print(f"{model_name}")
    print(f"  search   : {search_type}")
    print(f"  cv       : {cv}")
    print(f"  combos   : {n_combinations}")
    print(f"  total fits (combos x cv): {total_fits}")
    print(f"  scoring  : {scoring}")
    print()


## STEP 5 — WHAT GOT SAVED AND WHY


### -- three types of artifacts

the preprocessor (artifacts/preprocessor.pkl) — the fitted ColumnTransformer. this is needed at inference time to transform a new candidate's raw input into the same numeric format the models were trained on. without this, the models cannot be used on new data.

the models (artifacts/models/) — three pkl files, one per model, each containing the best estimator found by the hyperparameter search. the model with the best F1-macro on the test set is the winner and would be the one used in production.

the feature columns list (artifacts/feature_columns.pkl) — the exact list of column names that went into the preprocessor, in the exact order. this is needed at inference time to make sure the new candidate's data is in the same column order the preprocessor expects.


In [ ]:
from pathlib import Path

artifacts = [
    "../artifacts/preprocessor.pkl",
    "../artifacts/feature_columns.pkl",
    "../artifacts/models/logistic_regression.pkl",
    "../artifacts/models/random_forest.pkl",
    "../artifacts/models/xgboost.pkl",
]

print("Artifact check:")
for path in artifacts:
    exists = Path(path).exists()
    size_kb = round(Path(path).stat().st_size / 1024, 1) if exists else 0
    status = f"EXISTS ({size_kb} KB)" if exists else "MISSING"
    print(f"  {path.split('/')[-1]:<35} {status}")


## STEP 6 — RESULTS SUMMARY


In [ ]:
metrics_path = "../reports/metrics/metrics.csv"

if Path(metrics_path).exists():
    metrics_df = pd.read_csv(metrics_path)
    print("Model comparison — test set results:")
    print()
    print(metrics_df.to_string(index=False))
    print()
    best = metrics_df.sort_values(["F1-macro", "ROC-AUC"], ascending=False).iloc[0]
    print(f"Winner: {best['Model']} with F1-macro={best['F1-macro']} and ROC-AUC={best['ROC-AUC']}")
else:
    print("metrics.csv not found. Run python main.py --stage train first.")


## SUMMARY — THE FULL PIPELINE IN ONE PLACE

1. load raw CSV, remove duplicates, fill missing values, cap outliers — src/preprocessing.py
2. create 7 engineered features from the cleaned data — src/feature_engineering.py
3. stratified 80/20 train/test split so class proportions are preserved in both sets
4. fit ColumnTransformer on train only — OrdinalEncoder + OneHotEncoder + StandardScaler
5. apply SMOTE on the encoded training data only to fix class imbalance — never on test
6. run GridSearchCV for Logistic Regression and RandomizedSearchCV for RF and XGBoost with cv=4 and f1_macro scoring
7. evaluate each best model on the untouched test set — accuracy, F1-macro, ROC-AUC, FPR, FNR
8. save all three models, the preprocessor, plots, metrics CSV and summary.md

every decision in this pipeline was made to prevent data leakage, handle class imbalance honestly, and produce evaluation numbers that reflect real-world performance rather than optimistic in-sample scores.
